# Car Price Prediction using Linear Regression
Research AI Task 6 - Pure NumPy & Pandas Analytical Solution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv("CarPrice_Assignment.csv")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
df.head()

In [ ]:
numeric_features = [col for col in df.select_dtypes(include=np.number).columns if col not in ["price", "car_ID"]]
categorical_features = [col for col in df.select_dtypes(include="object").columns if col != "CarName"]
X_df = pd.get_dummies(df[numeric_features + categorical_features], drop_first=True, dtype=float)
X = X_df.to_numpy()
y = df["price"].to_numpy()
print("Number of features:", X.shape[1])

In [ ]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0, ddof=0)
keep = std > 0
X_train = (X_train[:, keep] - mean[keep]) / std[keep]
X_test = (X_test[:, keep] - mean[keep]) / std[keep]
X_train = np.c_[np.ones(len(X_train)), X_train]
X_test = np.c_[np.ones(len(X_test)), X_test]
print("Processed features count:", X_train.shape[1] - 1)

In [ ]:
theta, residuals, rank, s = np.linalg.lstsq(X_train, y_train, rcond=None)
print("Model trained successfully via analytical least squares.")

In [ ]:
test_predictions = X_test @ theta
mse = np.mean((test_predictions - y_test) ** 2)
rmse = np.sqrt(mse)
r2 = 1 - np.sum((y_test - test_predictions) ** 2) / np.sum((y_test - y_test.mean()) ** 2)
print("=== Performance Metrics ===")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, test_predictions, color='royalblue', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Fit')
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Car Prices")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
residuals = y_test - test_predictions
plt.figure(figsize=(8, 5))
plt.scatter(test_predictions, residuals, color='purple', alpha=0.7, edgecolors='k')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel("Predicted Prices")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot - Homoscedasticity Check")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20, color='teal', edgecolor='black', alpha=0.7)
plt.xlabel("Prediction Error (Residuals)")
plt.ylabel("Frequency")
plt.title("Distribution of Test Residuals")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()